# VocEd Lab 7 — Feature Engineering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/Labs/Lab7_feature_engineering.ipynb)

Lab 6 ended at a wall: every rule that looks at one pixel's colour is capped by the **colour
ceiling**, and our classifiers were already there.  The errors were pale patches inside nuclei and
dark grains in the cytoplasm — mistakes that only a pixel's **neighbourhood** can explain.

So in this lab we give each pixel more columns in the table.  Next to its R, G and B, every
pixel gets numbers describing its surroundings — **feature maps** made with the filters of
Chapter 7: blurs, a median, Sobel edges, local texture and Gabor stripes.  The same random forest
as Lab 6 then learns from all of them.

**Textbook:** [Chapter 7 — Feature Engineering for Image Segmentation](https://obooks.tech/imageprocessing/chapter7.html).

---

### How the puzzles work 🧩🤔

- **🧩 Code puzzles:** replace every `___` and run the cell — a ✅ means you got it.
- **🤔 Think puzzles:** answer in the ✏️ cell below the question.  There is no check: discuss
  your answer with your group.

Some cells take a few minutes; they say so.

## 0. Setup — Get the Data

The same images and split as Labs 3–6.  We also pick the same 450 training pixels per image as
Labs 5 and 6, so the only thing that changes is the columns.

In [ ]:
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

X = np.stack([np.load(f"imagedata/X/{i}.npy") for i in range(200)])   # (200, 3, 256, 256)
y = np.stack([np.load(f"imagedata/y/{i}.npy") for i in range(200)])   # (200, 256, 256)

nuc_px   = (y == 2).sum(axis=(1, 2))
quartile = np.digitize(nuc_px, np.percentile(nuc_px, [25, 50, 75]))
trainval_idx, test_idx = train_test_split(np.arange(200), test_size=0.2, stratify=quartile, random_state=42)
train_idx, val_idx = train_test_split(trainval_idx, test_size=40, stratify=quartile[trainval_idx], random_state=42)
assert list(test_idx[:5]) == [121, 152, 81, 4, 141], "Not the same test images as Labs 2–6"

def dice_score(pred, target, label):
    pred_mask   = (pred   == label)
    target_mask = (target == label)
    denominator = pred_mask.sum() + target_mask.sum()
    if denominator == 0:
        return 1.0
    return 2 * (pred_mask & target_mask).sum() / denominator

def image_dice(pred, mask):
    return (dice_score(pred, mask, 1) + dice_score(pred, mask, 2)) / 2      # cytoplasm and nucleus

def nc_ratio(mask):
    return (mask == 2).sum() / max((mask == 1).sum(), 1)

# The same 450 training pixels per image as Labs 5 and 6
rng = np.random.default_rng(0)
chosen = {i: rng.choice(65536, 450, replace=False) for i in train_idx}

**Scores so far, on the 40 validation images:** Lab 4 thresholds + clean-up 0.812; Lab 6 random
forest and MLP about 0.82 — at the colour ceiling.

## 1. Feature Maps

A **feature map** is an image the same size as the original, where each pixel holds one number
about that pixel's neighbourhood.  Most are made by sliding a small window (a **kernel**) over the
grayscale image.

| Feature | What it measures around a pixel | Book |
|---|---|---|
| `R`, `G`, `B` | the pixel's own colour | [§7.4](https://obooks.tech/imageprocessing/chapter7.html#raw-intensity-and-color-channel-maps) |
| `blur2`, `blur8` | average brightness nearby (Gaussian, σ = 2 and 8): *is this inside a dark region?* | [§7.5](https://obooks.tech/imageprocessing/chapter7.html#gaussian-blur-and-mean-blur) |
| `median5` | median brightness in a 5 × 5 window — ignores lone grains | [§7.6](https://obooks.tech/imageprocessing/chapter7.html#median-and-rank-filters) |
| `sobel` | how fast brightness changes: *is there an edge here?* | [§7.7](https://obooks.tech/imageprocessing/chapter7.html#sobel-scharr-gradient-magnitude) |
| `std5` | how much brightness varies in a 5 × 5 window: *is it grainy?* | |
| `gabor0`, `gabor90` | response to fine horizontal and vertical stripes — texture | [§7.8](https://obooks.tech/imageprocessing/chapter7.html#gabor-filters) |

### 🧩 Puzzle 1 — Sobel edge strength

`sobel(image, axis=0)` measures how fast brightness changes going *down* the rows; `axis=1`,
going *across* the columns.  The edge strength combines both: `√(gx² + gy²)`, which is
`np.hypot(gx, gy)`.  We smooth first (σ = 1) so single noisy pixels do not count as edges.

### 🧩 Puzzle 2 — Local standard deviation

The standard deviation is `√(mean of squares − square of mean)`.  `uniform_filter(a, 5)` is the
mean of `a` over each 5 × 5 window.  (Rounding can make the difference a hair below zero, so we
clip it at 0 before the square root.)

<details><summary><b>💡 Hint 1</b> (click to open)</summary>

Puzzle 1: the second direction is the other axis.

</details>

<details><summary><b>💡 Hint 2</b> (click to open)</summary>

Puzzle 2: the *mean of squares* is the window mean of `gray` squared — `gray ** 2`.

</details>

In [ ]:
from scipy.ndimage import gaussian_filter, median_filter, sobel, uniform_filter
from skimage.filters import gabor

names = ['R', 'G', 'B', 'blur2', 'blur8', 'median5', 'sobel', 'std5', 'gabor0', 'gabor90']

def feature_maps(img):
    gray   = 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]
    smooth = gaussian_filter(gray, 1)
    return {
        'R': img[0], 'G': img[1], 'B': img[2],
        'blur2':   gaussian_filter(gray, 2),
        'blur8':   gaussian_filter(gray, 8),
        'median5': median_filter(gray, 5),
        'sobel':   np.hypot(sobel(smooth, axis=0), sobel(smooth, axis=___)),
        'std5':    np.sqrt(np.maximum(uniform_filter(___, 5) - uniform_filter(gray, 5) ** 2, 0)),
        'gabor0':  gabor(gray, frequency=0.25, theta=0)[0],
        'gabor90': gabor(gray, frequency=0.25, theta=np.pi / 2)[0],
    }

maps7 = feature_maps(X[7])
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, name in zip(axes.flat, names):
    ax.imshow(maps7[name], cmap='gray')
    ax.set_title(name)
    ax.axis('off')
plt.show()

# ── Check ──
gray7 = 0.299 * X[7][0] + 0.587 * X[7][1] + 0.114 * X[7][2]
assert maps7['sobel'][100:110, 100:110].min() >= 0 and abs(maps7['sobel'] - np.hypot(sobel(gaussian_filter(gray7, 1), 0), sobel(gaussian_filter(gray7, 1), 1))).max() < 1e-6, "Combine the change down the rows and across the columns"
assert abs(maps7['std5'][128, 128] - gray7[126:131, 126:131].std()) < 1e-4, "The window standard deviation"
print("✅ Puzzles 1 and 2 solved!")

### 🤔 Think 1 — Looking at the maps

1. In which maps do the nuclei stand out most clearly?  In which do the **borders** stand out?
2. A dark grain in the cytoplasm is dark in `G` but not in `blur8`.  Why?  How could a classifier
   use the two together?
3. Chapter 7 also builds **GLCM** texture features.  For a per-pixel GLCM, a co-occurrence matrix
   is counted for the window around *every* pixel — 65,536 windows per image.  Why did we leave it
   out of a lab that must run in a few minutes?

📖 Book: [§7.9.4 Using GLCM for Segmentation](https://obooks.tech/imageprocessing/chapter7.html#using-glcm-for-segmentation)

✏️ **Your answer:** *(double-click this cell and type here)*

## 2. From Maps to a Table

The classifier still sees one **row per pixel** — now with ten columns instead of three.  Compute
the maps for all 200 images first.  *(About 15 seconds.)*

### 🧩 Puzzle 3 — Stack the columns

Each map, flattened, is one column of 65,536 values.  `np.stack` puts the columns side by side —
along which axis?

📖 Book: [§7.1 Why Feature Maps Matter](https://obooks.tech/imageprocessing/chapter7.html#why-feature-maps-matter)

<details><summary><b>💡 Hint</b> (click to open)</summary>

The result should have shape (65536, 10): rows are axis 0, columns are axis 1.

</details>

In [ ]:
maps = {i: feature_maps(X[i]) for i in range(200)}

def feature_table(i, columns):
    return np.stack([maps[i][c].ravel() for c in columns], axis=___)

table7 = feature_table(7, names)
print("Feature table for image 7:", table7.shape)

# ── Check ──
assert table7.shape == (65536, 10), "One row per pixel, one column per feature"
print("✅ Puzzle 3 solved!")

## 3. Which Features Help?  An Ablation

We train Lab 6's random forest (100 trees, `min_samples_leaf = 100`) on different groups of
columns and score each on the 40 validation images.  *(About 3 minutes.)*

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def train_forest(columns):
    X_tab = np.vstack([feature_table(i, columns)[chosen[i]] for i in train_idx])
    y_tab = np.concatenate([y[i].reshape(-1)[chosen[i]] for i in train_idx])
    return RandomForestClassifier(n_estimators=100, min_samples_leaf=100, random_state=0).fit(X_tab, y_tab)

def predict_mask(forest, columns, i):
    return forest.predict(feature_table(i, columns)).reshape(256, 256)

groups = {
    'colour only':           ['R', 'G', 'B'],
    '+ neighbourhood':       ['R', 'G', 'B', 'blur2', 'blur8', 'median5'],
    '+ edges':               ['R', 'G', 'B', 'blur2', 'blur8', 'median5', 'sobel'],
    '+ texture (all ten)':   names,
    'neighbourhood, no colour': ['blur2', 'blur8', 'median5', 'sobel', 'std5', 'gabor0', 'gabor90'],
}

forests, val_scores = {}, {}
for group, columns in groups.items():
    forests[group] = train_forest(columns)
    val_scores[group] = np.array([image_dice(predict_mask(forests[group], columns, i), y[i]) for i in val_idx])
    print(f"{group:26s} {len(columns):2d} features   validation Dice {val_scores[group].mean():.4f}")

### 🧩 Puzzle 4 — Rank the features

Which features did the all-ten forest rely on?  Sort the importances from largest to smallest.

📖 Book: [§5.7 Method 3 — Random Forest](https://obooks.tech/imageprocessing/chapter5.html#method-3-random-forest)

<details><summary><b>💡 Hint</b> (click to open)</summary>

`np.argsort` gives the positions that sort an array from smallest to largest.  `[::-1]` reverses an array.

</details>

In [ ]:
importances = forests['+ texture (all ten)'].feature_importances_
order = np.___(importances)[::-1]

for position in order:
    print(f"  {names[position]:8s} {importances[position]:.3f}  " + "█" * int(importances[position] * 100))

# ── Check ──
assert importances[order[0]] == importances.max() and importances[order[-1]] == importances.min(), "Largest first"
print("✅ Puzzle 4 solved!")

### 🤔 Think 2 — What the ablation says

1. How much did the neighbourhood features add to colour?  Edges?  Texture?  Compare these gains
   with the step from Lab 2 to Lab 3.
2. `sobel` has one of the lowest importances.  An edge map is bright on *both sides* of a border.
   Why does knowing "there is an edge here" not tell the forest which class a pixel is?
3. The forest without any colour columns scores about the same as colour alone.  What does that tell
   you about how much information the *neighbourhood brightness* carries compared with the colour?
4. Seven hand-made features for the price of three hours of reading Chapter 7, and the gain is
   small.  Was it the wrong eight?  How would you find better ones — by hand, or some other way?

📖 Book: [§7.10 From Feature Engineering to CNNs](https://obooks.tech/imageprocessing/chapter7.html#from-feature-engineering-to-cnns)

✏️ **Your answer:** *(double-click this cell and type here)*

## 4. The Final Test — Once

The all-ten forest on the 40 test images.  Lab 4's clean-up worked on thresholded masks — does it
help the forest's masks too?  We test both, but we have to decide *before* looking at the test
images: on validation first.

In [ ]:
from skimage.measure import label
from scipy.ndimage import binary_fill_holes

def remove_specks(pred, min_size=100):                 # from Lab 4
    labels    = label(pred == 2, connectivity=2)
    areas     = np.bincount(labels.ravel())
    small_ids = np.where(areas < min_size)[0]
    out = pred.copy()
    out[np.isin(labels, small_ids) & (labels > 0)] = 1
    return out

def fill_holes(pred):                                  # from Lab 4
    filled = binary_fill_holes(pred == 2)
    out = pred.copy()
    out[filled & (pred == 1)] = 2
    return out

forest = forests['+ texture (all ten)']
cleaned_val = np.mean([image_dice(fill_holes(remove_specks(predict_mask(forest, names, i))), y[i]) for i in val_idx])
print(f"Validation Dice: forest {val_scores['+ texture (all ten)'].mean():.4f},  forest + clean-up {cleaned_val:.4f}")

In [ ]:
nc_true = np.array([nc_ratio(y[i]) for i in test_idx])
candidates = {
    'forest, all ten features':  lambda i: predict_mask(forest, names, i),
    '  + Lab 4 clean-up':        lambda i: fill_holes(remove_specks(predict_mask(forest, names, i))),
}

print(f"{'':28s} test Dice   N/C R²   N/C within ±0.05")
for name, model in candidates.items():
    preds   = [model(i) for i in test_idx]
    dice    = np.mean([image_dice(p, y[i]) for p, i in zip(preds, test_idx)])
    nc_pred = np.array([nc_ratio(p) for p in preds])
    r2      = 1 - np.sum((nc_pred - nc_true) ** 2) / np.sum((nc_true - nc_true.mean()) ** 2)
    close   = np.mean(np.abs(nc_pred - nc_true) < 0.05)
    print(f"{name:28s} {dice:.4f}    {r2:7.3f}   {close:.0%}")

print()
print("Earlier labs, on the same 40 test images:")
print("  Lab 4 specks, fill              0.772     -0.465   35%")

### 🤔 Think 3 — Hand-made features, honestly

1. Did the features beat Lab 4 and Lab 6 on the test images?  By how much?
2. Did the clean-up help the forest on validation?  On test?  Why might steps designed for a
   threshold's mistakes do less for a forest that already sees the neighbourhood?
3. Six labs in, test Dice is still around 0.77 and the N/C R² is still poor.  Every model so far has
   been given its view of the neighbourhood by us — a fixed blur, a fixed window.  What would a model
   that can learn *which* neighbourhood pattern matters need to be able to do?

📖 Book: [§7.10 From Feature Engineering to CNNs](https://obooks.tech/imageprocessing/chapter7.html#from-feature-engineering-to-cnns)

✏️ **Your answer:** *(double-click this cell and type here)*

## Wrap-up

**Key takeaways:**
- **A feature map is a filter's answer for every pixel.**  Stacking maps as columns turns
  "neighbourhood" into numbers a classifier can use.
- **Ablation tells you what each group of features is worth.**  Here the neighbourhood, edge and
  texture features each added only a little to colour.
- **An edge map is not a label.**  Sobel says *a border is near*, not *which side* — that is why it
  belongs among a classifier's features, not in a threshold pipeline, and why even there it ranks low.
- **Hand-made features are guesses.**  Each fixes a kernel we chose.  The next step is to let the model
  learn its own kernels from the data.

---

### Group discussion 👥

Split the work, then share your numbers:

| Person | Change, then re-run Sections 2–3 |
|---|---|
| A | Add `'blur16': gaussian_filter(gray, 16)` to the maps and to `names` |
| B | Gabor with `frequency=0.1` instead of 0.25 (coarser stripes) |
| C | Add `'scharr'` from `skimage.filters.scharr(smooth)` and remove `'sobel'` |

1. Did any new feature earn a high importance?  Did it move validation Dice?
2. Is there any sign that bigger neighbourhoods (person A) matter more than finer texture (person B)?
3. If one feature costs ten times more to compute and adds 0.002 Dice, would you keep it?